# 🧱 Lab: Prompt A/B Evaluation

**Module 9: Is It Any Good? Quality, Latency, Cost** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Name** a handful of production failure modes that evaluation should watch for
2. **Run** prompt V1 vs V2 on the gold set
3. **Write** a one-paragraph ship / don't-ship decision with a self-check rubric
4. **Tag** `v4`

## Concepts Covered

| Concept | Gloss | From |
|---------|-------|------|
| Prompt versioning | Messages, roles, params are structured data with a version label | mini-typed-prompt-parser |
| Gold set | Small verified labelled sample | mini-score-against-gold-set |
| Precision and recall | Per-class beats headline | mini-score-against-gold-set |
| Baseline comparison | Judge against a simple reference | mini-score-against-gold-set |
| Evidence-based decision | "I would ship V2 because ___, accepting ___" | This lab |
| AI failure modes | Out-of-schema labels, drift, rate limits, silent fallback | This lab |

## Prerequisites

- **mini-score-against-gold-set**: Baseline + gold set scoring
- **mini-latency-and-cost**: p50 / p95 and cost per 1,000


---
## Production failure modes (short lecture)

Evaluation must catch breakages that stay **silent** in production. Nothing crashes; the output just gets worse. Watch for out-of-schema labels, silent fallback to a default answer, rate limits, input drift, and a gold set too small to trust a 2-point accuracy bump. The ship decision at the end names the ones to monitor.

## The Scenario

You have an LLM-based system that classifies support tickets into three categories: **billing**, **technical**, and **account**. Your team wrote a prompt (V1) that's been running in production. A colleague proposed a rewritten prompt (V2) that they claim is better.

**Your job:** Run both prompts against the same gold set, measure quality + latency + cost, and write a defensible recommendation — ship V2, keep V1, or neither.

This is **prompt versioning** in practice: treating prompt changes like code changes and evaluating them with evidence before deploying.

In [1]:
from IPython.display import display, Markdown
def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## Step 1 — The Gold Set and Two Prompt Versions

We have a 40-row gold set of support tickets with human-verified labels. Both prompt V1 and V2 were run against every ticket. We also logged latency and token usage for each call (as we learned in `mini-latency-and-cost`).

In a real project, you'd run the actual LLM calls. Here we use the recorded results so the lab is reproducible and doesn't require an API key.

In [2]:
from collections import Counter

# Gold set: 40 tickets with human-verified labels (18 billing, 13 technical, 9 account)
gold_labels = ["billing"] * 18 + ["technical"] * 13 + ["account"] * 9

# V1: short zero-shot prompt. Good at billing, weaker on technical/account.
v1_predictions = (
    ["billing"] * 15 + ["technical"] * 2 + ["account"]          # true billing
    + ["technical"] * 8 + ["billing"] * 3 + ["account"] * 2     # true technical
    + ["account"] * 5 + ["billing"] * 3 + ["technical"]         # true account
)

# V2: longer prompt with examples. Better on technical/account, slightly worse on billing.
v2_predictions = (
    ["billing"] * 14 + ["technical"] * 2 + ["account"] * 2      # true billing
    + ["technical"] * 11 + ["billing"] + ["account"]            # true technical
    + ["account"] * 7 + ["billing"] + ["technical"]             # true account
)

print(f"Gold set size: {len(gold_labels)} tickets")
print(f"Label distribution: {dict(Counter(gold_labels))}")

Gold set size: 40 tickets
Label distribution: {'billing': 18, 'technical': 13, 'account': 9}


## Step 2 — Majority-Class Baseline

Recall from `mini-score-against-gold-set`: before comparing V1 and V2, we need to know the accuracy of the *dumbest possible system* — one that always predicts the most common label. Both prompts must beat this to justify their existence.

In [3]:
def accuracy(true, predicted):
    """Fraction of predictions that match the true labels."""
    return sum(t == p for t, p in zip(true, predicted)) / len(true)

majority_class = Counter(gold_labels).most_common(1)[0][0]
baseline_acc = accuracy(gold_labels, [majority_class] * len(gold_labels)) * 100
v1_acc = accuracy(gold_labels, v1_predictions) * 100
v2_acc = accuracy(gold_labels, v2_predictions) * 100

print(f"Majority-class baseline (always '{majority_class}'): {baseline_acc:.1f}%")
print(f"Prompt V1 accuracy: {v1_acc:.1f}%  (margin: {v1_acc - baseline_acc:+.1f} pp)")
print(f"Prompt V2 accuracy: {v2_acc:.1f}%  (margin: {v2_acc - baseline_acc:+.1f} pp)")

Majority-class baseline (always 'billing'): 45.0%
Prompt V1 accuracy: 70.0%  (margin: +25.0 pp)
Prompt V2 accuracy: 80.0%  (margin: +35.0 pp)


## Step 3 — Confusion Matrices

Accuracy tells us *how many* errors each prompt makes. Confusion matrices tell us *what kind*. We'll build side-by-side matrices so we can see exactly where each prompt struggles.

In [4]:
def confusion_matrix(true, predicted, labels):
    """Build a confusion matrix: matrix[true][predicted] = count."""
    matrix = {t: {p: 0 for p in labels} for t in labels}
    for t, p in zip(true, predicted):
        matrix[t][p] += 1
    return matrix

def format_cm(cm, labels, title):
    """Markdown table: one row per true label."""
    table = f"#### {title}\n\n| True \\ Predicted | " + " | ".join(labels) + " |\n"
    table += "|" + "---|" * (len(labels) + 1) + "\n"
    for t in labels:
        table += f"| **{t}** | " + " | ".join(str(cm[t][p]) for p in labels) + " |\n"
    return table

labels = sorted(set(gold_labels))  # ['account', 'billing', 'technical']
cm_v1 = confusion_matrix(gold_labels, v1_predictions, labels)
cm_v2 = confusion_matrix(gold_labels, v2_predictions, labels)

md(format_cm(cm_v1, labels, "Prompt V1") + "\n" +
   format_cm(cm_v2, labels, "Prompt V2") + "\n" +
   "*Diagonal = correct predictions.*")

#### Prompt V1

| True \ Predicted | account | billing | technical |
|---|---|---|---|
| **account** | 5 | 3 | 1 |
| **billing** | 1 | 15 | 2 |
| **technical** | 2 | 3 | 8 |

#### Prompt V2

| True \ Predicted | account | billing | technical |
|---|---|---|---|
| **account** | 7 | 1 | 1 |
| **billing** | 2 | 14 | 2 |
| **technical** | 1 | 1 | 11 |

*Diagonal = correct predictions.*

## Step 4 — Per-Class Precision and Recall

Recall from `mini-score-against-gold-set`:
- **Precision** = of everything predicted as class X, what fraction truly was X?
- **Recall** = of everything truly class X, what fraction did the system catch?

We compute these for both prompts and display them side by side.

In [5]:
def precision_recall(cm, labels):
    """Compute per-class precision and recall from a confusion matrix."""
    metrics = {}
    for cls in labels:
        tp = cm[cls][cls]
        predicted_as_cls = sum(cm[t][cls] for t in labels)
        actually_cls = sum(cm[cls][p] for p in labels)
        precision = tp / predicted_as_cls if predicted_as_cls > 0 else 0.0
        recall = tp / actually_cls if actually_cls > 0 else 0.0
        metrics[cls] = {"precision": precision, "recall": recall, "support": actually_cls}
    return metrics

m_v1 = precision_recall(cm_v1, labels)
m_v2 = precision_recall(cm_v2, labels)

# One row per class: precision / recall for each prompt
table = "### Per-Class Precision / Recall: V1 vs V2\n\n| Class | V1 P / R | V2 P / R | Support |\n|---|---|---|---|\n"
for cls in labels:
    v1, v2 = m_v1[cls], m_v2[cls]
    table += (f"| **{cls}** | {v1['precision']:.0%} / {v1['recall']:.0%} "
              f"| {v2['precision']:.0%} / {v2['recall']:.0%} | {v1['support']} |\n")
md(table)

### Per-Class Precision / Recall: V1 vs V2

| Class | V1 P / R | V2 P / R | Support |
|---|---|---|---|
| **account** | 62% / 56% | 70% / 78% | 9 |
| **billing** | 71% / 83% | 88% / 78% | 18 |
| **technical** | 73% / 62% | 79% / 85% | 13 |


## Step 5 — Latency and Cost Comparison

Quality alone doesn't determine whether to ship. Prompt V2 is longer (it includes examples), so it uses more tokens and may have higher latency. Let's measure that.

We use logged latency and token data, just as we did in `mini-latency-and-cost`.

In [6]:
import random
random.seed(99)

def simulate_perf_logs(n, avg_prompt_tok, latency_base):
    """Simulate n logged calls: mostly normal latency plus a slow tail."""
    logs = []
    for _ in range(n):
        # About 1 call in 10 is slow. p95 exists to catch this long tail.
        if random.random() < 0.1:
            lat = latency_base * random.uniform(3, 6)
        else:
            lat = latency_base * random.uniform(0.7, 1.3)
        logs.append({
            "latency_ms": lat,
            "prompt_tokens": avg_prompt_tok + random.randint(-40, 40),
            "completion_tokens": random.randint(5, 25),
        })
    return logs

# V2's longer prompt (with examples) means more tokens and higher latency
logs_v1 = simulate_perf_logs(40, avg_prompt_tok=180, latency_base=220)
logs_v2 = simulate_perf_logs(40, avg_prompt_tok=310, latency_base=340)

def percentile(values, p):
    s = sorted(values)
    return s[min(int(len(s) * p / 100), len(s) - 1)]

# gpt-4o-mini pricing, in $ per 1M tokens
PRICE_IN = 0.15
PRICE_OUT = 0.60

def compute_stats(logs):
    lats = [e["latency_ms"] for e in logs]
    avg_pt = sum(e["prompt_tokens"] for e in logs) / len(logs)
    avg_ct = sum(e["completion_tokens"] for e in logs) / len(logs)
    cost_per_call = (avg_pt * PRICE_IN + avg_ct * PRICE_OUT) / 1_000_000
    return {
        "p50": percentile(lats, 50),
        "p95": percentile(lats, 95),
        "avg_prompt_tokens": avg_pt,
        "avg_completion_tokens": avg_ct,
        "cost_per_1000": cost_per_call * 1000,
    }

v1_stats = compute_stats(logs_v1)
v2_stats = compute_stats(logs_v2)

md(f"""### Latency & Cost: V1 vs V2

| Metric | V1 | V2 |
|--------|-----|-----|
| p50 latency | {v1_stats['p50']:.0f} ms | {v2_stats['p50']:.0f} ms |
| p95 latency | {v1_stats['p95']:.0f} ms | {v2_stats['p95']:.0f} ms |
| Avg prompt tokens | {v1_stats['avg_prompt_tokens']:.0f} | {v2_stats['avg_prompt_tokens']:.0f} |
| Avg completion tokens | {v1_stats['avg_completion_tokens']:.0f} | {v2_stats['avg_completion_tokens']:.0f} |
| **Cost per 1,000** | **${v1_stats['cost_per_1000']:.4f}** | **${v2_stats['cost_per_1000']:.4f}** |
""")

### Latency & Cost: V1 vs V2

| Metric | V1 | V2 |
|--------|-----|-----|
| p50 latency | 244 ms | 346 ms |
| p95 latency | 285 ms | 1356 ms |
| Avg prompt tokens | 177 | 311 |
| Avg completion tokens | 15 | 15 |
| **Cost per 1,000** | **$0.0358** | **$0.0555** |


## Step 6 — Full Comparison Dashboard

Let's pull everything together — quality, latency, and cost — into one view. This is the evidence table that drives the ship decision.

In [7]:
# One row per class (precision / recall), then latency and cost
class_rows = ""
for cls in labels:
    class_rows += (f"| {cls} P / R | — | {m_v1[cls]['precision']:.0%} / {m_v1[cls]['recall']:.0%} "
                   f"| {m_v2[cls]['precision']:.0%} / {m_v2[cls]['recall']:.0%} |\n")

md(f"""### 📊 Prompt V1 vs V2: Full Comparison

| Metric | Baseline | Prompt V1 | Prompt V2 |
|--------|----------|-----------|-----------|
| **Accuracy** | {baseline_acc:.1f}% | {v1_acc:.1f}% | {v2_acc:.1f}% |
{class_rows}| p50 / p95 latency | — | {v1_stats['p50']:.0f} / {v1_stats['p95']:.0f} ms | {v2_stats['p50']:.0f} / {v2_stats['p95']:.0f} ms |
| **Cost per 1,000** | — | ${v1_stats['cost_per_1000']:.4f} | ${v2_stats['cost_per_1000']:.4f} |

*Gold set size: {len(gold_labels)} tickets, so treat all metrics as directional.*
""")

### 📊 Prompt V1 vs V2: Full Comparison

| Metric | Baseline | Prompt V1 | Prompt V2 |
|--------|----------|-----------|-----------|
| **Accuracy** | 45.0% | 70.0% | 80.0% |
| account P / R | — | 62% / 56% | 70% / 78% |
| billing P / R | — | 71% / 83% | 88% / 78% |
| technical P / R | — | 73% / 62% | 79% / 85% |
| p50 / p95 latency | — | 244 / 285 ms | 346 / 1356 ms |
| **Cost per 1,000** | — | $0.0358 | $0.0555 |

*Gold set size: 40 tickets, so treat all metrics as directional.*


## Step 7 — Failure Mode Check

Before writing the ship decision, recall from the failure-modes lecture at the top of this lab that we should ask: **how could each prompt break silently in production?**

Let's check whether either prompt produced any hallucinated labels (labels outside the allowed set) or refusals during our evaluation run.

> 🧩 **AI failure modes.** The silent breakages from the video; now we check for one.

In [8]:
VALID_LABELS = {"billing", "technical", "account"}

def check_silent_failures(predictions, version_name):
    """Check for hallucinated or invalid labels."""
    invalid = [p for p in predictions if p not in VALID_LABELS]
    if invalid:
        print(f"⚠️  {version_name}: {len(invalid)} invalid label(s) found: {invalid}")
    else:
        print(f"✅  {version_name}: all {len(predictions)} predictions are valid labels")
    return invalid

check_silent_failures(v1_predictions, "Prompt V1")
check_silent_failures(v2_predictions, "Prompt V2")

print("\n💡 In production, validate every response against VALID_LABELS.")

✅  Prompt V1: all 40 predictions are valid labels
✅  Prompt V2: all 40 predictions are valid labels

💡 In production, validate every response against VALID_LABELS.


## Step 8 — Write the Ship Decision

An **evidence-based ship decision** is not "V2 looks better, let's ship it." It's a structured document that:

1. **States the question** clearly
2. **Presents the evidence** (the metrics we just computed)
3. **Acknowledges the trade-offs** — what gets better AND what gets worse
4. **Makes a recommendation** with its accepted risks
5. **States the caveats** (sample size, conditions under which to revisit)

This is **prompt versioning** in action: we treat the prompt change like a code change and gate it behind evidence.

> 🧩 **Ship / don't-ship decision.** The launch call from the video, now filled with our numbers.

In [9]:
# Key deltas that the decision cites
tech_gain = m_v2["technical"]["recall"] - m_v1["technical"]["recall"]
acct_gain = m_v2["account"]["recall"] - m_v1["account"]["recall"]
billing_change = m_v2["billing"]["recall"] - m_v1["billing"]["recall"]
latency_change = v2_stats["p50"] - v1_stats["p50"]
cost_change_pct = (v2_stats["cost_per_1000"] - v1_stats["cost_per_1000"]) / v1_stats["cost_per_1000"] * 100

md(f"""### 📋 Ship Decision: Prompt V1 → V2

| Metric | V1 | V2 | Change |
|--------|----|----|--------|
| Accuracy (baseline {baseline_acc:.1f}%) | {v1_acc:.1f}% | {v2_acc:.1f}% | {v2_acc - v1_acc:+.1f} pp |
| technical recall | {m_v1['technical']['recall']:.0%} | {m_v2['technical']['recall']:.0%} | {tech_gain:+.0%} |
| account recall | {m_v1['account']['recall']:.0%} | {m_v2['account']['recall']:.0%} | {acct_gain:+.0%} |
| billing recall | {m_v1['billing']['recall']:.0%} | {m_v2['billing']['recall']:.0%} | {billing_change:+.0%} |
| p50 latency | {v1_stats['p50']:.0f} ms | {v2_stats['p50']:.0f} ms | {latency_change:+.0f} ms |
| Cost per 1,000 | ${v1_stats['cost_per_1000']:.4f} | ${v2_stats['cost_per_1000']:.4f} | {cost_change_pct:+.0f}% |

**I would ship V2 because** technical recall rises {tech_gain:+.0%} and account recall {acct_gain:+.0%}, lifting accuracy from {v1_acc:.1f}% to {v2_acc:.1f}% (both well above the {baseline_acc:.1f}% baseline). **I accept** a {billing_change:+.0%} change in billing recall, {latency_change:+.0f} ms at p50, and ~{cost_change_pct:.0f}% higher cost per 1,000 calls. Billing still has the highest recall, and the extra cost is small in absolute terms. **Caveat:** with only {len(gold_labels)} gold rows these are directional estimates. After the switch we will monitor billing recall, out-of-schema labels, and input drift, and grow the gold set to 80–100 rows.
""")

### 📋 Ship Decision: Prompt V1 → V2

| Metric | V1 | V2 | Change |
|--------|----|----|--------|
| Accuracy (baseline 45.0%) | 70.0% | 80.0% | +10.0 pp |
| technical recall | 62% | 85% | +23% |
| account recall | 56% | 78% | +22% |
| billing recall | 83% | 78% | -6% |
| p50 latency | 244 ms | 346 ms | +103 ms |
| Cost per 1,000 | $0.0358 | $0.0555 | +55% |

**I would ship V2 because** technical recall rises +23% and account recall +22%, lifting accuracy from 70.0% to 80.0% (both well above the 45.0% baseline). **I accept** a -6% change in billing recall, +103 ms at p50, and ~55% higher cost per 1,000 calls. Billing still has the highest recall, and the extra cost is small in absolute terms. **Caveat:** with only 40 gold rows these are directional estimates. After the switch we will monitor billing recall, out-of-schema labels, and input drift, and grow the gold set to 80–100 rows.


---
## Self-check rubric

Tag `v4` only after the report passes this list:

- [ ] Gold set size and the sample-size caveat are stated
- [ ] Both prompts are compared against the majority-class baseline
- [ ] Per-class precision and recall are shown, not just accuracy
- [ ] p50 / p95 latency and cost per 1,000 are reported
- [ ] Accepted trade-offs and what to monitor are named
- [ ] The prompt + model version is recorded (next step)

## Step 9 — Version Control: Tag Your Evaluation (`v4`)

Every ship decision should record **exactly** which configuration produced the results. This isn't about Git commits (though those help). It's the minimum version metadata that makes an evaluation reproducible.

In [10]:
from datetime import datetime

eval_tag = {
    "eval_id": "eval-2024-001",
    "date": datetime.now().strftime("%Y-%m-%d"),
    "prompt_version": "V2",
    "model": "gpt-4o-mini",
    "gold_set_version": "gold-v1 (40 rows)",
    "decision": "SHIP V2 for background routing",
}

tag_rows = "\n".join(f"| **{k}** | {v} |" for k, v in eval_tag.items())
md("### 🏷️ Evaluation Tag (`v4`)\n\nSave this tag next to your report.\n\n| Field | Value |\n|---|---|\n" + tag_rows)

### 🏷️ Evaluation Tag (`v4`)

Save this tag next to your report.

| Field | Value |
|---|---|
| **eval_id** | eval-2024-001 |
| **date** | 2026-10-18 |
| **prompt_version** | V2 |
| **model** | gpt-4o-mini |
| **gold_set_version** | gold-v1 (40 rows) |
| **decision** | SHIP V2 for background routing |

## 🎯 Summary

### What You Built

You built a complete prompt A/B evaluation pipeline that compares two prompt versions against the same gold set, measures quality (accuracy, per-class precision and recall, confusion matrices), latency (p50/p95), and cost — then synthesizes the evidence into a defensible ship/don't-ship recommendation.

### Key Takeaways

1. **Prompt versioning** — treat prompt changes like code changes: never swap prompts in production without running both versions against the same gold set and comparing the evidence
2. **Baseline comparison** — both prompt versions must beat the majority-class baseline; if they don't, the AI system isn't adding value regardless of which version "wins"
3. **Per-class metrics over accuracy** — overall accuracy can hide that one prompt is much better at minority classes; precision and recall per class reveal *where* each version succeeds and fails
4. **Quality + latency + cost** — a better prompt that's too slow or too expensive might not be worth shipping; always evaluate all three dimensions together
5. **Evidence-based decision** — a ship recommendation states the question, presents the evidence, acknowledges what gets worse, names the accepted trade-offs, and flags the sample-size caveats